In [1]:
import os, torch
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:128'
torch.cuda.empty_cache()
import os, sys
os.environ["TORCH_NVML_DISABLED"] = "1"
os.chdir("/scratch/jq2uw/MME/instruct_vlm_edit")

repo_root = "/scratch/jq2uw/MME/instruct_vlm_edit"
if repo_root not in sys.path:
    sys.path.append(repo_root)

from revlm.config_utils import *
from revlm.dataset import *
from revlm.models import *
import argparse

In [2]:
cfg_path = os.path.join(repo_root, "revlm", "config", "config.yaml")

# Simulate CLI overrides
args = argparse.Namespace(
    config=cfg_path,
    editor="ft",  # should become config.editor._name
    inner_params=["transformer.h.0.mlp.c_fc.weight"],  # -> config.model.inner_params
    dataset_name="aokvqa",#"fvqa",  # -> config.experiment.dataset_name
    model_name="llava",
)

config = configure_args(args, config_path=cfg_path)
# config.experiment.streaming = True
config

namespace(batch_size=1,
          n_iter=100,
          max_n_edits=5000,
          seed=42,
          device='cuda',
          ckpt_dir=None,
          dropout=None,
          res_dir='results/ft/llava-1.5-7b-hf/aokvqa',
          model=namespace(name='llava-hf/llava-1.5-7b-hf',
                          class_name='VQAModel',
                          pt=None,
                          inner_params=['model.language_model.layers.0.self_attn.q_proj.weight'],
                          processor_class=None,
                          tokenizer_class=None,
                          temperature=1.0),
          editor=namespace(_name='ft', edit_lr='1e-4'),
          experiment=namespace(task='mci',
                               dataset_name='aokvqa',
                               split='train'))

In [3]:
vlm = VQAModel(config)

/scratch/jq2uw/conda_ex/conda_venv/revlm/lib/python3.10/site-packages/transformers/models/auto/modeling_auto.py:2291: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [4]:
# ds = get_dataset(config, split="train")
# print(f"Edit dataset size: {len(ds)}")
# ds.set_dataloader(task="mci", shuffle_choices=True, seed=333, batch_size=64)

# # ds.data = random.sample(ds.data, 10)
# # ds.task_generate(vlm)
# # ds.data[0]

In [5]:
edit_dataset = VQADataset(config)
edit_dataset.data = random.sample(edit_dataset.data, 10)
# print(f"Edit dataset size: {len(edit_dataset)}")
edit_dataset.set_dataloader()
edit_dataset.task_generate(vlm)

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

mc

In [6]:
# edit_dataset.task_generate(batch, vlm)
edit_dataset.data[0]
edit_dataset.task_engineer.eval(edit_dataset)

{'image': 'data/images/aokvqa/train2017/000000421562.jpg', 'question': 'The last four letters seen in the background are all found in what word?', 'answer': 'pizzeria', 'rationale': 'The letters seen in the background are found in pizzeria.', 'choices': 'pizzeria; loquacious; quash; sublime', 'idx_choices': '(A) pizzeria\n(B) loquacious\n(C) quash\n(D) sublime', 'idx': 0, 'gold': {'choices': {'str': '(C) sublime\n(B) quash\n(D) pizzeria\n(A) loquacious', 'ls': [('C', 'sublime'), ('B', 'quash'), ('D', 'pizzeria'), ('A', 'loquacious')]}, 'label': 'pizzeria', 'label_letter': 'D', 'label_train': '(D) pizzeria'}, 'prompt': 'Choose A/B/C/D from the options. The last four letters seen in the background are all found in what word? Options: (C) sublime\n(B) quash\n(D) pizzeria\n(A) loquacious', 'pred': {'answer': 'D', 'letter_text': 'D', 'label_text': None, 'label_scores': {'sublime': {'avg_nll': 6.350007057189941, 'sum_nll': 12.700014114379883, 'num_tokens': 2, 'prob': 0.33979158377608587}, 'q

{'letter_text': {'accuracy': 0.8,
  'n': 10,
  'confusion_matrix': [[2, 0, 0, 0],
   [0, 0, 1, 0],
   [0, 1, 2, 0],
   [0, 0, 0, 4]]},
 'letter_maxprob': {'accuracy': 0.8,
  'n': 10,
  'confusion_matrix': [[2, 0, 0, 0],
   [1, 0, 0, 0],
   [0, 1, 2, 0],
   [0, 0, 0, 4]]},
 'label_text': {'accuracy': 0.6,
  'n': 5,
  'confusion_matrix': [[0, 1, 0, 0],
   [0, 2, 0, 0],
   [0, 0, 1, 1],
   [0, 0, 0, 0]]},
 'label_maxprob': {'accuracy': 0.7,
  'n': 10,
  'confusion_matrix': [[1, 1, 0, 0],
   [0, 2, 0, 0],
   [1, 0, 3, 1],
   [0, 0, 0, 1]]}}

In [7]:

# g_labels = [g['label'] for g in batch['golds']]
# label_words = [g['choices']['ls'] for g in batch['golds']]

# s = vlm.score_choices(images, prompts, label_words)
# for ex in edit_dataset.data:
#     s = vlm.score_choices_single(ex['image'], ex['prompt'], ex['gold']['choices']['ls'])
#     break
# s

In [8]:
edit_dataset.set_dataloader(task="mci", shuffle_choices=True, seed=333, batch_size=64)
for batch in edit_dataset.loader:
    images = batch['images']
    prompts = batch['prompts']
    golds = batch['golds']
    break

In [7]:
g_letters = [g['label_letter'] for g in batch['golds']]
g_labels = [g['label'] for g in batch['golds']]

use_letter = False
letter_lists = [[c[0] for c in g['choices']['ls']] for g in batch['golds']]
text_lists   = [[c[1] for c in g['choices']['ls']] for g in batch['golds']]
label_words = letter_lists if use_letter else text_lists

s = vlm.score_choices(images, prompts, text_lists)
c = vlm.score_choices(images, prompts, letter_lists)

KeyboardInterrupt: 

In [15]:
correct = 0
for i in range(len(s)):
    d = s[i]
    best_label = max(d, key=lambda k: d[k]['prob'])
    pred_text = (best_label[1] if isinstance(best_label, (tuple, list)) else best_label)
    gold_text = str(g_letters[i]) if use_letter else str(g_labels[i])
    if str(pred_text).strip().lower() == gold_text.strip().lower():
        correct += 1
acc = correct / len(s) if s else 0.0
print(f"Batch accuracy: {acc:.4f} ({correct}/{len(s)})")


Batch accuracy: 0.5625 (36/64)


In [20]:
model = vlm
for ex in edit_dataset.data:
    ex['pred'] = {}
    # score-based generation
    label_texts = [choice for _, choice in ex['gold']['choices']['ls']]
    label_letters = [ltr for ltr, _ in ex['gold']['choices']['ls']]
    ex['pred']['label_scores'] = model.score_choices_single(ex['image'], ex['prompt'], label_texts)
    ex['pred']['letter_scores'] = model.score_choices_single(ex['image'], ex['prompt'], label_letters)
    ex['pred']['label_maxprob'] = max(ex['pred']['label_scores'], key=lambda k: ex['pred']['label_scores'][k]['prob'])
    ex['pred']['letter_maxprob'] = max(ex['pred']['letter_scores'], key=lambda k: ex['pred']['letter_scores'][k]['prob'])
        
    break

edit_dataset.data[0]

{'image': 'data/images/aokvqa/train2017/000000299207.jpg',
 'question': 'What is the man by the bags awaiting?',
 'answer': 'cab',
 'rationale': 'A train would not be on the street, he would not have luggage waiting for a delivery, and the skateboarder is there and not paying attention to him so a cab is the only possible answer.',
 'choices': 'skateboarder; train; delivery; cab',
 'idx_choices': '(A) skateboarder\n(B) train\n(C) delivery\n(D) cab',
 'gold': {'choices': {'str': '(D) cab\n(A) skateboarder\n(B) train\n(C) delivery',
   'ls': [('D', 'cab'),
    ('A', 'skateboarder'),
    ('B', 'train'),
    ('C', 'delivery')]},
  'label': 'cab',
  'label_letter': 'D'},
 'prompt': 'Choose A/B/C/D from the options. What is the man by the bags awaiting? Options: (D) cab\n(A) skateboarder\n(B) train\n(C) delivery',
 'pred': {'label_scores': {'cab': {'avg_nll': 13.287845611572266,
    'sum_nll': 13.287845611572266,
    'num_tokens': 1,
    'prob': 0.9715263101961199},
   'skateboarder': {'avg_

In [15]:
max(s, key=lambda k: s[k]['prob'])

'cab'

In [27]:
correct = 0
for i in range(len(s)):
    d = s[i]
    best_label = max(d, key=lambda k: d[k]['prob'])
    pred_text = (best_label[1] if isinstance(best_label, (tuple, list)) else best_label)
    gold_text = str(g_letters[i]) if use_letter else str(g_labels[i])
    if str(pred_text).strip().lower() == gold_text.strip().lower():
        correct += 1
acc = correct / len(s) if s else 0.0
print(f"Batch accuracy: {acc:.4f} ({correct}/{len(s)})")


Batch accuracy: 0.5625 (36/64)


In [28]:
for i in range(len(s)):
    print(str(g_letters[i]) + " " + str(g_labels[i]))
    print(label_words[i])
    d = s[i]
    best_label = max(d, key=lambda k: d[k]['prob'])
    best_prob = d[best_label]['prob']
    print(best_label, best_prob)

D cab
['cab', 'skateboarder', 'train', 'delivery']
cab 0.9715263101961199
A office
['motel', 'cafe', 'office', 'outside']
office 0.9745132594046344
B farmer
['waiter', 'musician', 'farmer', 'cashier']
farmer 0.9853290096666447
B airport workers
['police', 'firemen', 'postal workers', 'airport workers']
airport workers 0.9965004258427326
B two
['three', 'two', 'four', 'one']
two 0.39956702224749086
A theater
['cinema', 'internet', 'tv', 'theater']
theater 0.9986941837914322
B happy
['sad', 'happy', 'curious', 'scared']
happy 0.5067913738813271
D downpour
['sprinkle', 'drizzle', 'average', 'downpour']
sprinkle 0.5447613534451305
D bow
['bow', 'ribbon', 'camera', 'rag']
ribbon 0.541178081652471
D turn right
['reverse course', 'turn left', 'turn right', 'drive straight']
drive straight 0.8111273371928673
B bike
['bike', 'jogged', 'motorcycle', 'police car']
bike 0.9933827391266273
B human
['bird', 'cow', 'human', 'elephant']
human 0.8253056430859117
A uk
['usa', 'cuba', 'uk', 'mexico']
uk 

In [23]:
from revlm.metrics import QA_metrics_text, QA_metrics_loss, QA_metrics_nli_bi

# Load test split and prepare loader
test_dataset = get_dataset(config, split="train")
import random
subsample = 90
if len(test_dataset) > subsample:
    test_dataset.data = random.sample(test_dataset.data, subsample)
test_dataset.set_dataloader(
    task="qa",
    with_rationale=False,
    shuffle_choices=False,
    batch_size=32,
)

# Run metrics
res_text = QA_metrics_text(vlm, test_dataset)
res_loss = QA_metrics_loss(vlm, test_dataset)
res_nli = QA_metrics_nli_bi(vlm, test_dataset)



print("QA Text parse:", res_text)
print("QA Log-loss:", res_loss)
print("QA NLI Bi:", res_nli)

ImportError: cannot import name 'extract_choice_pairs' from 'revlm.dataset.utils' (/sfs/weka/scratch/jq2uw/MME/instruct_vlm_edit/revlm/dataset/utils/__init__.py)

In [5]:
# Evaluate MCQ on FVQA test split
from revlm.metrics.mcq import MCQ_metrics_text, MCQ_metrics_score, MCQ_metrics_classifier
test_dataset.set_dataloader(
    task="mcq",
    with_rationale=False,
    shuffle_choices=False,
    batch_size=32,
)

# Run metrics
res_text = MCQ_metrics_text(vlm, test_dataset)
res_score = MCQ_metrics_score(vlm, test_dataset, score_by_letter=True)
res_score_option = MCQ_metrics_score(vlm, test_dataset, score_by_letter=False)
res_cls = MCQ_metrics_classifier(vlm, test_dataset)



print("MCQ Text parse:", res_text)
print("MCQ Log-loss:", res_score)
print("MCQ Log-loss (option):", res_score_option)
print("MCQ Classifier:", res_cls)


MCQ Text parse: {'accuracy': 0.8571428571428571, 'n': 14, 'confusion_matrix': [[1, 0, 0, 0], [0, 2, 0, 0], [0, 1, 5, 1], [0, 0, 0, 4]]}
MCQ Log-loss: {'accuracy': 0.7888888888888889, 'n': 90, 'confusion_matrix': [[19, 0, 1, 1], [4, 10, 1, 0], [5, 0, 21, 0], [5, 1, 1, 21]]}
MCQ Log-loss (option): {'accuracy': 0.4444444444444444, 'n': 90, 'confusion_matrix': [[15, 3, 1, 2], [5, 5, 4, 1], [10, 3, 12, 1], [12, 3, 5, 8]]}
MCQ Classifier: {'accuracy': 0.8, 'n': 90, 'confusion_matrix': [[18, 0, 2, 1], [5, 10, 0, 0], [1, 2, 22, 1], [3, 2, 1, 22]]}


In [7]:
test_dataset.set_dataloader(
    task="mcq",
    with_rationale=False,
    shuffle_choices=True,
    batch_size=32,
)

# Run metrics
res_text = MCQ_metrics_text(vlm, test_dataset)
res_score = MCQ_metrics_score(vlm, test_dataset)
res_cls = MCQ_metrics_classifier(vlm, test_dataset)

print("MCQ Text parse:", res_text)
print("MCQ Log-loss:", res_score)
print("MCQ Classifier:", res_cls)


MCQ Text parse: {'accuracy': 0.84, 'n': 25, 'confusion_matrix': [[2, 1, 1, 0], [0, 5, 0, 0], [0, 1, 8, 1], [0, 0, 0, 6]]}
MCQ Log-loss: {'accuracy': 0.7555555555555555, 'n': 90, 'confusion_matrix': [[20, 0, 0, 1], [5, 10, 0, 0], [8, 0, 17, 1], [5, 1, 1, 21]]}
MCQ Classifier: {'accuracy': 0.7111111111111111, 'n': 90, 'confusion_matrix': [[19, 1, 0, 1], [6, 9, 0, 0], [5, 1, 18, 2], [5, 2, 3, 18]]}


In [ ]:
edit_dataset = get_dataset(config)
print(f"Edit dataset size: {len(edit_dataset)}")
edit_dataset.shuffle_choices()
edit_dataset.data[0]


choices = []
imgs = []
questions = []
for i in range(20):
    ex = edit_dataset.data[i]
    choices.append(ex["choices"])
    imgs.append(ex["image"]) # read image fomr ex["image_path"]
    questions.append("Choose A/B/C/D based on the image."+ex["question"] + ex['choices'])

In [ ]:

with torch.no_grad():
    ans = vlm.generate(images=imgs, prompts=questions, max_new_tokens=100)

ans